In [94]:
import pandas as pd

# =========================
# 1. 文件路径
# =========================
clinical_file = "../data/case/LungCancer/tcga/luad_cptac_gdc_clinical_data.tsv"
tp53_file = "../data/case/LungCancer/tcga/tp53_mut_clinical.txt"
output_file = "../data/case/LungCancer/tcga/luad_cptac_gdc_clinical_data_with_TP53.tsv"

# =========================
# 2. 读取数据
# =========================
clinical = pd.read_csv(clinical_file, sep="\t", dtype=str)
tp53_mut = pd.read_csv(tp53_file, sep="\t", comment="#", dtype=str)

# =========================
# 3. 标准化样本ID格式
# =========================
clinical["Sample ID"] = clinical["Sample ID"].astype(str).str.strip()
tp53_mut["SAMPLE_ID"] = tp53_mut["SAMPLE_ID"].astype(str).str.strip()

# 去掉缺失和字符串nan
clinical = clinical[clinical["Sample ID"].notna()]
tp53_mut = tp53_mut[tp53_mut["SAMPLE_ID"].notna()]

clinical = clinical[clinical["Sample ID"].str.lower() != "nan"]
tp53_mut = tp53_mut[tp53_mut["SAMPLE_ID"].str.lower() != "nan"]

# =========================
# 4. 取交集样本
# =========================
common_samples = set(clinical["Sample ID"]) & set(tp53_mut["SAMPLE_ID"])

# 只保留交集样本
clinical_intersection = clinical[clinical["Sample ID"].isin(common_samples)].copy()

# =========================
# 5. 添加 TP53 突变状态列
# =========================
clinical_intersection["TP53_status"] = clinical_intersection["Sample ID"].apply(
    lambda x: "TP53_Mut" if x in common_samples else "TP53_WT"
)

# 如果你想强制用 0/1，也可以改成：
# clinical_intersection["TP53_status"] = clinical_intersection["Sample ID"].apply(
#     lambda x: 1 if x in common_samples else 0
# )

# =========================
# 6. 保存结果
# =========================
clinical_intersection.to_csv(output_file, sep="\t", index=False)

# =========================
# 7. 输出统计信息
# =========================
print("原始 clinical 样本数:", clinical.shape[0])
print("tp53_mut_clinical 样本数:", tp53_mut.shape[0])
print("交集样本数:", len(common_samples))
print("\nTP53_status 分布:")
print(clinical_intersection["TP53_status"].value_counts(dropna=False))
print(f"\n结果已保存到: {output_file}")

原始 clinical 样本数: 231
tp53_mut_clinical 样本数: 241
交集样本数: 231

TP53_status 分布:
TP53_status
TP53_Mut    231
Name: count, dtype: int64

结果已保存到: ../data/case/LungCancer/tcga/luad_cptac_gdc_clinical_data_with_TP53.tsv


In [95]:
import pandas as pd

# 输入文件
clinical_file = "../data/case/LungCancer/tcga/luad_cptac_gdc_clinical_data_with_TP53.tsv"
data_file = "../data/case/LungCancer/tcga/tpm.txt"


# 1. 读取 clinical
clinical = pd.read_csv(clinical_file, sep="\t", dtype=str)
#clinical["PATIENT_ID"] = clinical["PATIENT_ID"] + "-01"

data = pd.read_csv(data_file, sep="\t", comment="#", dtype=str,index_col=0)


common_samples = [x for x in data.columns if x in set(clinical['Sample ID'])]


# 表达矩阵取交集
data_filtered = data[common_samples]

# 临床表取交集
clinical_filtered = clinical[clinical['Sample ID'].isin(common_samples)]


In [98]:
clinical_filtered["Overall Survival Status"] = clinical_filtered["Overall Survival Status"].str.split(":").str[0].astype(int)

ValueError: cannot convert float NaN to integer

In [91]:
clinical_filtered.to_csv('../data/case/LungCancer/tcga/processed_clinical.csv')